In [ ]:
import ast
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from chef_classifier.data import (
    clean_training_data,
    create_train_val_split,
    load_training_data,
)
from chef_classifier.evaluation import calculate_accuracy
from chef_classifier.features import combine_text_fields
from chef_classifier.models import build_tfidf_svc_pipeline

In [9]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test-no-labels.csv"

In [ ]:
train = load_training_data(TRAIN_PATH)
test = pd.read_csv(TEST_PATH, sep=";")

In [ ]:
train_clean = clean_training_data(train)
train_clean.shape

In [ ]:
train_df, val_df = create_train_val_split(train_clean)

In [ ]:
X_train = combine_text_fields(train_df, ["description"])
X_val = combine_text_fields(val_df, ["description"])

model = build_tfidf_svc_pipeline()
model.fit(X_train, train_df["chef_id"])

predictions = model.predict(X_val)

accuracy = calculate_accuracy(val_df["chef_id"], predictions)
accuracy

In [ ]:
svc = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("classifier", SVC(kernel="linear")),
    ]
)
svc.fit(X_train, train_df["chef_id"])

svc_predictions = svc.predict(X_val)
accuracy_score(val_df["chef_id"], svc_predictions)

In [ ]:
print(classification_report(val_df["chef_id"], predictions))

In [17]:
train_df["description"].duplicated().sum(), val_df["description"].duplicated().sum()

(np.int64(18), np.int64(3))

In [18]:
val_df["description"].isin(train_df["description"]).sum()

np.int64(7)

In [19]:
duplicate_description_mask = val_df["description"].isin(train_df["description"])

duplicate_description_mask.sum()

np.int64(7)

In [20]:
clean_val_accuracy = accuracy_score(
    val_df.loc[~duplicate_description_mask, "chef_id"],
    predictions[~duplicate_description_mask],
)

clean_val_accuracy

0.7305084745762712

In [21]:
overlap_accuracy = accuracy_score(
    val_df.loc[duplicate_description_mask, "chef_id"],
    predictions[duplicate_description_mask],
)

overlap_accuracy

1.0

- TF-IDF features extracted only from the description field perform surprisingly strongly:
  - linear-kernel SVC: 71.9%
  - LinearSVC: 73.4%
- Seven validation examples have descriptions that also occur in the training subset.
- The model predicts all seven of these overlapping descriptions correctly.
- Excluding those overlapping examples reduces validation accuracy only slightly, from 73.4% to 73.1%.
- Therefore, repeated descriptions introduce a small amount of leakage but do not explain the high overall performance.

In [22]:
val_results = val_df.copy()

val_results["predicted_chef"] = predictions
val_results["correct"] = (
    val_results["chef_id"] == val_results["predicted_chef"]
)

val_results.head()

,chef_id,recipe_name,data,tags,steps,description,ingredients,n_ingredients,predicted_chef,correct
1000,4470,carrot salad with raspberry vinaigrette,25/05/2002,"['15-minutes-or-less', 'time-to-make', 'course...","['mince parsley in the processor , stalks remo...",this is a do in advance salad the flavor impro...,"['fresh parsley', 'lite olive oil', 'raspberry...",9,4470,True
2069,4470,greek potato skins,18/11/2001,"['weeknight', 'time-to-make', 'course', 'main-...","['heat oven 400f , prick the potatoes and bake...","you can do most of this recipe ahead, have you...","['potatoes', 'olive oil', 'feta cheese', 'oreg...",7,4470,True
1948,4470,individual mini shrimp quiches,08/05/2002,"['60-minutes-or-less', 'time-to-make', 'course...",['bake the tart shells in a 375f oven for abou...,these are great for brunch or as an appetizer ...,"['frozen mini tart shells', 'shrimp', 'eggs', ...",8,4470,True
1537,5060,toasted mushroom roll ups,02/07/2002,"['60-minutes-or-less', 'time-to-make', 'course...","['in a large frypan , heat 1 tablespoon of the...","tasty morsels, a great appetizer!! (1 hour chi...","['margarine', 'mushroom', 'onion', 'sweet red ...",9,5060,True
2658,3288,basil shrimp and feta pasta,11/03/2002,"['30-minutes-or-less', 'time-to-make', 'course...",['heat the olive oil in a skillet over medium-...,i love shrimp and feta so this is a real winne...,"['olive oil', 'yellow onion', 'garlic', 'prawn...",12,8688,False


In [23]:
val_results["correct"].value_counts()

correct
True     438
False    159
Name: count, dtype: int64

In [24]:
val_results["correct"].mean()

np.float64(0.7336683417085427)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    val_results["chef_id"],
    val_results["predicted_chef"],
)

In [26]:
confusion_table = pd.crosstab(
    val_results["chef_id"],
    val_results["predicted_chef"],
    rownames=["True chef"],
    colnames=["Predicted chef"],
)

confusion_table

Predicted chef,1533,3288,4470,5060,6357,8688
True chef,,,,,,
1533,46,13,9,8,2,2
3288,11,52,12,4,2,9
4470,7,6,134,12,0,1
5060,5,3,10,83,2,4
6357,5,4,3,3,54,4
8688,2,2,6,7,1,69


In [27]:
errors = val_results[~val_results["correct"]].copy()

errors.shape

(159, 10)

In [30]:
errors[
    [
        "chef_id",
        "predicted_chef",
        "recipe_name",
        "description",
        "ingredients",
        "tags",
    ]
].sample(10, random_state=42)

,chef_id,predicted_chef,recipe_name,description,ingredients,tags
1463,3288,1533,cheddar crab quiche,"for a completely vegetarian quiche, leave the ...","['frozen hash browns', 'white crab meat', 'red...","['weeknight', 'time-to-make', 'course', 'main-..."
2602,4470,1533,low fat jambalaya,very tasty and lower fat than regular jambalaya,"['bay scallops', 'turkey kielbasa', 'canola oi...","['60-minutes-or-less', 'time-to-make', 'course..."
477,3288,4470,cheese and hot dog burritos,not gourmet but a way to make a burrito fun fo...,"['flour tortillas', 'hot dogs', 'cheddar chees...","['15-minutes-or-less', 'time-to-make', 'course..."
104,5060,8688,lemon cream cheese pie with berries,special dessert for company or special occasio...,"['cream cheese', 'fat-free sweetened condensed...","['weeknight', 'time-to-make', 'course', 'main-..."
2728,1533,3288,orange pineapple coconut smoothie,this is sooooooo good.,"['orange juice', 'pineapple juice', 'coconut m...","['15-minutes-or-less', 'time-to-make', 'course..."
34,4470,1533,iced mandarin orange tea,this is for a hot day!,"['tea bags', 'boiling water', 'sugar', 'fresh ...","['lactose', '15-minutes-or-less', 'time-to-mak..."
1073,6357,1533,umm ali,courtesy: boulvar,"['puff pastry', 'milk', 'cinnamon', 'sultanas'...","['60-minutes-or-less', 'time-to-make', 'course..."
611,1533,5060,wheat germ muffins,these taste very nutty.,"['flour', 'baking powder', 'salt', 'honey', 'w...","['60-minutes-or-less', 'time-to-make', 'course..."
886,3288,8688,sofrito basico,this sofrito recipe is prepared as the first s...,"['yellow onion', 'garlic', 'green bell pepper'...","['30-minutes-or-less', 'time-to-make', 'course..."
2788,5060,3288,marinated flank steak with mustard sauce,"works well for a dinner party, barbecue party ...","['buttermilk', 'mustard', 'garlic cloves', 'lo...","['weeknight', 'time-to-make', 'course', 'main-..."


In [31]:
confusion_pairs = (
    errors.groupby(["chef_id", "predicted_chef"])
    .size()
    .sort_values(ascending=False)
)

confusion_pairs.head(10)

chef_id  predicted_chef
1533     3288              13
4470     5060              12
3288     4470              12
         1533              11
5060     4470              10
3288     8688               9
1533     4470               9
         5060               8
4470     1533               7
8688     5060               7
dtype: int64

In [32]:
pair_errors = errors[
    (errors["chef_id"] == 3288)
    & (errors["predicted_chef"] == 5060)
]

pair_errors[
    [
        "recipe_name",
        "description",
        "ingredients",
        "tags",
    ]
].head(10)

,recipe_name,description,ingredients,tags
1596,garlic chive pesto,this spread is excellent on smoked salmon and ...,"['garlic sprouts', 'fresh flat-leaf parsley', ...","['15-minutes-or-less', 'time-to-make', 'course..."
1007,pressure cooker beef barley vegetable soup,"this is a cross between a soup and a stew, and...","['lean ground beef', 'crushed tomatoes', 'wate...","['60-minutes-or-less', 'time-to-make', 'course..."
1233,grilled pork tenderloins with wasabi sauce,leftovers make great sandwiches. serve over ri...,"['pork tenderloin', 'soy sauce', 'frozen orang...","['60-minutes-or-less', 'time-to-make', 'course..."
1456,tamale lentil casserole,from best of baking. i love the taste of tamal...,"['olive oil', 'onion', 'green bell pepper', 'g...","['weeknight', 'time-to-make', 'course', 'main-..."


In [33]:
correct_3288 = val_results[
    (val_results["chef_id"] == 3288)
    & (val_results["correct"])
]

correct_5060 = val_results[
    (val_results["chef_id"] == 5060)
    & (val_results["correct"])
]

In [34]:
val_results["description_length"] = (
    val_results["description"].str.len()
)

val_results.groupby("correct")[
    "description_length"
].describe()

,count,mean,std,min,25%,50%,75%,max
correct,,,,,,,,
False,159.0,111.069182,70.506048,5.0,55.50,93.0,156.5,386.0
True,438.0,171.696347,146.148874,3.0,75.25,127.5,215.5,970.0


In [35]:
val_results["description_words"] = (
    val_results["description"].str.split().str.len()
)

val_results.groupby("correct")[
    "description_words"
].describe()

,count,mean,std,min,25%,50%,75%,max
correct,,,,,,,,
False,159.0,20.446541,13.120015,1.0,10.0,18.0,30.0,77.0
True,438.0,32.038813,28.015701,1.0,14.0,23.5,40.0,185.0


In [36]:
accuracy_by_chef = (
    val_results.groupby("chef_id")["correct"]
    .mean()
    .sort_values()
)

accuracy_by_chef

chef_id
1533    0.575000
3288    0.577778
6357    0.739726
5060    0.775701
8688    0.793103
4470    0.837500
Name: correct, dtype: float64

Classification difficulty varies substantially between chefs:

- chef 1533: 57.5%
- chef 3288: 57.8%
- chef 6357: 74.0%
- chef 5060: 77.6%
- chef 8688: 79.3%
- chef 4470: 83.8%

The errors are not uniformly distributed. Some of the most frequent confusion directions are, this suggests that some chefs use more similar vocabulary or description styles than others.

Incorrectly classified recipes tend to have shorter descriptions:

- incorrect predictions: about 20.4 words on average
- correct predictions: about 32.0 words on average

The corresponding median lengths are approximately 18** and 23.5 words.

This suggests that short or generic descriptions provide less lexical and stylistic evidence for identifying the chef.

Several misclassified examples have very short or generic descriptions, such as phrases equivalent to "this is so good", "for a hot day", or short comments about serving suggestions.

In such cases, useful identifying information may exist in other fields such as:

- recipe_name
- ingredients
- tags
- steps

The next experiments should therefore test whether adding these fields improves performance, particularly for chefs 1533 and 3288.

To measure the contribution of each field, the following experiments keep the validation split and LinearSVC classifier fixed while changing only the text fields used to build the TF-IDF representation.

In [ ]:
def evaluate_text_fields(fields: list[str]) -> float:
    train_text = combine_text_fields(train_df, fields)
    val_text = combine_text_fields(val_df, fields)

    model = build_tfidf_svc_pipeline()
    model.fit(train_text, train_df["chef_id"])

    predictions = model.predict(val_text)

    return calculate_accuracy(val_df["chef_id"], predictions)

In [39]:
experiments = {
    "description": ["description"],
    "description + recipe_name": ["description", "recipe_name"],
    "description + ingredients": ["description", "ingredients"],
    "description + tags": ["description", "tags"],
    "description + steps": ["description", "steps"],
    "all text fields": [
        "description",
        "recipe_name",
        "ingredients",
        "tags",
        "steps",
    ],
}

results = {}

for name, fields in experiments.items():
    results[name] = evaluate_text_fields(fields)

results

{'description': 0.7336683417085427,
 'description + recipe_name': 0.7671691792294807,
 'description + ingredients': 0.7654941373534339,
 'description + tags': 0.9045226130653267,
 'description + steps': 0.7872696817420436,
 'all text fields': 0.8760469011725294}

In [40]:
results_df = (
    pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=["accuracy"],
    )
    .sort_values("accuracy", ascending=False)
)

results_df

,accuracy
description + tags,0.904523
all text fields,0.876047
description + steps,0.787270
description + recipe_name,0.767169
description + ingredients,0.765494
description,0.733668


In [41]:
tag_only_accuracy = evaluate_text_fields(["tags"])
tag_only_accuracy

0.8207705192629816

In [42]:
train_df.groupby("chef_id")["tags"].apply(
    lambda x: x.head(3).tolist()
)

chef_id
1533    [['15-minutes-or-less', 'time-to-make', 'cours...
3288    [['weeknight', 'time-to-make', 'course', 'main...
4470    [['15-minutes-or-less', 'time-to-make', 'cours...
5060    [['course', 'main-ingredient', 'cuisine', 'nor...
6357    [['time-to-make', 'course', 'main-ingredient',...
8688    [['time-to-make', 'course', 'main-ingredient',...
Name: tags, dtype: object

In [ ]:
train_tags = train_df[["chef_id", "tags"]].copy()

train_tags["tags"] = train_tags["tags"].apply(ast.literal_eval)

train_tags.head()

In [49]:
exploded_tags = (
    train_tags
    .explode("tags")
    .reset_index(drop=True)
)

exploded_tags.head()

,chef_id,tags
0,8688,time-to-make
1,8688,course
2,8688,main-ingredient
3,8688,preparation
4,8688,desserts


In [50]:
tag_counts = (
    exploded_tags.groupby(["chef_id", "tags"])
    .size()
    .reset_index(name="count")
)

tag_counts.head()

,chef_id,tags,count
0,1533,1-day-or-more,3
1,1533,15-minutes-or-less,100
2,1533,3-steps-or-less,66
3,1533,30-minutes-or-less,97
4,1533,4-hours-or-less,42


In [51]:
top_tags_by_chef = (
    tag_counts.sort_values(
        ["chef_id", "count"],
        ascending=[True, False],
    )
    .groupby("chef_id")
    .head(15)
)

top_tags_by_chef

,chef_id,tags,count
185,1533,preparation,322
246,1533,time-to-make,320
69,1533,course,317
77,1533,dietary,307
143,1533,main-ingredient,292
...,...,...,...
1604,8688,stove-top,96
1377,8688,60-minutes-or-less,95
1622,8688,vegetables,93
1372,8688,15-minutes-or-less,89


In [52]:
tag_chef_table = pd.crosstab(
    exploded_tags["tags"],
    exploded_tags["chef_id"],
)

tag_chef_table

chef_id,1533,3288,4470,5060,6357,8688
tags,,,,,,
1-day-or-more,3,3,12,5,9,8
15-minutes-or-less,100,46,118,70,38,89
3-steps-or-less,66,42,113,28,18,47
30-minutes-or-less,97,73,192,114,42,62
4-hours-or-less,42,106,123,99,85,74
...,...,...,...,...,...,...
wings,1,0,1,0,1,2
winter,3,57,4,78,11,3
yams-sweet-potatoes,0,14,4,5,2,2


In [53]:
tag_dominance = pd.DataFrame({
    "total_count": tag_chef_table.sum(axis=1),
    "dominant_chef": tag_chef_table.idxmax(axis=1),
    "dominant_count": tag_chef_table.max(axis=1),
})

tag_dominance["dominance_ratio"] = (
    tag_dominance["dominant_count"]
    / tag_dominance["total_count"]
)

In [54]:
tag_dominance[
    tag_dominance["total_count"] >= 10
].sort_values(
    ["dominance_ratio", "total_count"],
    ascending=[False, False],
).head(30)

,total_count,dominant_chef,dominant_count,dominance_ratio
tags,,,,
scandinavian,17,4470,17,1.000000
danish,14,4470,14,1.000000
british-columbian,243,5060,242,0.995885
pacific-northwest,100,5060,99,0.990000
ontario,69,1533,67,0.971014
indian,223,6357,212,0.950673
ramadan,13,6357,12,0.923077
curries,24,6357,22,0.916667
copycat,10,3288,9,0.900000


### Tag analysis

The high predictive performance of tags can be explained by strong associations between certain tags and individual chefs.

Several tags are almost exclusive to a single chef. Examples include:

- scandinavian: 17/17 occurrences belong to chef 4470
- danish: 14/14 belong to chef 4470
- british-columbian: 242/243 belong to chef 5060
- pacific-northwest: 99/100 belong to chef 5060
- ontario: 67/69 belong to chef 1533
- indian: 212/223 belong to chef 6357

Other tags such as ramadan, curries, gluten-free, asian, and oamc-freezer-make-ahead also show strong chef-specific concentration.

This indicates that the classifier is not relying only on linguistic writing style. It can also identify chefs from the types of recipes they tend to create and the categories associated with those recipes.

This appears to be a legitimate predictive signal in the supplied dataset, although it may also reflect dataset-specific author preferences. Therefore, very high local validation performance may not generalize equally well if the distribution of recipe categories changes in the test set.